# Privacy-Preserving Fraud Detection Using Machine Learning
### Academic Assessment: IA 1 — Problem Formulation and Preliminary Implementation
**Subject:** Security and Privacy for Machine Learning (Semester VII)  
**Total Marks:** 20 | **Target Dataset:** ULB Credit Card Fraud Detection (Kaggle)

---
### Academic Rubric Mapping
1. **Problem definition and module mapping (4 Marks):** Addresses class imbalance, privacy threat models, and maps to Modules 1–5.
2. **Conceptual understanding (4 Marks):** Rigorous formulations of MIA, DP ($\epsilon, \delta, C$), LDP, PPDM, and Compressive Privacy ($I(Z; S) - \beta I(Z; Y)$).
3. **Dataset identification and preliminary preparation (4 Marks):** Diagnostic inspection, class imbalance handling, leak-free scaling.
4. **Initial implementation or demonstration (4 Marks):** Baseline Random Forest, controlled MIA, and Differential Privacy sweep with IBM `diffprivlib`.
5. **Preliminary evaluation and documentation (4 Marks):** PR-AUC, Confusion Matrix, Attack AUC, and Privacy-Utility Pareto frontier.


## 0. Environment Setup & Reproducibility
Run this cell in **Google Colab** or locally in VS Code. It installs required privacy libraries (`diffprivlib`) and configures deterministic random seeds.


In [ ]:
# Google Colab installation helper (if running in Colab)
try:
    import google.colab
    IN_COLAB = True
    !pip install -q diffprivlib scikit-learn pandas numpy matplotlib seaborn
except ImportError:
    IN_COLAB = False

import os
import sys
import json
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Running in Colab: {IN_COLAB}")
print(f"Project Root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import diffprivlib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve
)

from configs.config import RANDOM_SEED, BASELINE_RF_PARAMS
from src.synthetic_data import generate_synthetic_fraud_data
from src.data_preprocessing import load_dataset, inspect_dataset, split_data, preprocess_features
from src.evaluation import compute_classification_metrics, plot_confusion_matrix, plot_precision_recall_curve, plot_roc_curve, plot_privacy_utility_curve

print(f"diffprivlib version: {diffprivlib.__version__}")
print(f"Random seed configured: {RANDOM_SEED}")


## 1. Dataset Acquisition & Diagnostic Inspection
We load the **ULB Credit Card Fraud Detection** dataset. If `data/creditcard.csv` is not present, our pipeline automatically uses a statistically realistic **synthetic verification dataset** with identical schema (5,000 transactions, 0.5% fraud) so all experiments run end-to-end immediately.


In [ ]:
df, provenance_tag = load_dataset()
inspection = inspect_dataset(df)

print(f"Dataset Provenance: {provenance_tag}")
print(f"Total Rows: {len(df):,}")
print(f"Fraudulent Transactions: {inspection['class_distribution']['fraudulent_1']} ({inspection['class_distribution']['fraud_prevalence_percentage']}%)")
print(f"Legitimate Transactions: {inspection['class_distribution']['legitimate_0']}")
print(f"Missing Values: {inspection['total_missing_values']}")
print(f"Duplicate Rows: {inspection['duplicate_records']}")

df.head(3)


## 2. Train/Validation/Test Split & Anti-Leakage Protocol
### Addressing the Imbalance & Split Protocol:
- **Stratified Split:** Guarantees that the ~0.17% (or 0.5% in synthetic test) fraud prevalence is preserved identically across training (70%), validation (10%), and testing (20%) sets.
- **Data Leakage Prevention:** Feature scalers (`RobustScaler` for transaction amounts) are fitted **strictly on the training data**, and then used to transform validation and test sets.


In [ ]:
train_df, test_df, val_df, split_meta = split_data(df, method="stratified")
X_train, X_test, X_val, y_train, y_test, y_val, feature_names, scaler = preprocess_features(
    train_df, test_df, val_df
)

print(f"Training set:   {X_train.shape[0]} rows ({y_train.sum()} frauds, {y_train.mean()*100:.3f}%)")
print(f"Validation set: {X_val.shape[0]} rows ({y_val.sum()} frauds, {y_val.mean()*100:.3f}%)")
print(f"Test set:       {X_test.shape[0]} rows ({y_test.sum()} frauds, {y_test.mean()*100:.3f}%)")
print(f"Total features: {len(feature_names)}")


## 3. Experiment A: Non-Private Baseline Fraud Detection
We train a **Random Forest Classifier** (`class_weight='balanced'`) to detect fraud.

### Why Accuracy Alone is Misleading:
In highly imbalanced datasets where 99.83% of transactions are legitimate, a naive model that predicts every transaction as `0` achieves **99.83% accuracy**, yet fails to detect a single fraud (Recall = 0.0).  
Therefore, **PR-AUC, Recall, Precision, and F1-score** are the primary authoritative evaluation metrics.


In [ ]:
rf_baseline = RandomForestClassifier(**BASELINE_RF_PARAMS)
rf_baseline.fit(X_train, y_train)

y_pred_base = rf_baseline.predict(X_test)
y_prob_base = rf_baseline.predict_proba(X_test)[:, 1]

baseline_metrics = compute_classification_metrics(y_test, y_pred_base, y_prob_base, "Baseline_RandomForest")
print("=== Baseline Performance Metrics ===")
for k, v in baseline_metrics.items():
    if k != "accuracy_critique":
        print(f"{k:20s}: {v}")
print("
Critique:", baseline_metrics["accuracy_critique"])


In [ ]:
# Diagnostic Visualizations: Confusion Matrix & PR Curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), dpi=120)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_base)
cax = ax1.matshow(cm, cmap="Blues", alpha=0.85)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax1.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                 color="white" if cm[i, j] > cm.max()/2 else "black", fontweight="bold")
fig.colorbar(cax, ax=ax1)
ax1.set_xticks([0, 1])
ax1.set_yticks([0, 1])
ax1.set_xticklabels(["Legitimate (0)", "Fraud (1)"])
ax1.set_yticklabels(["Legitimate (0)", "Fraud (1)"])
ax1.set_title("Baseline Confusion Matrix", fontweight="bold")

# Precision-Recall Curve
precs, recs, _ = precision_recall_curve(y_test, y_prob_base)
ap = average_precision_score(y_test, y_prob_base)
ax2.plot(recs, precs, color="#1f77b4", lw=2, label=f"Random Forest (PR-AUC = {ap:.4f})")
ax2.axhline(y_test.mean(), color="red", linestyle="--", label=f"Random Guess (Base Rate={y_test.mean():.4f})")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curve", fontweight="bold")
ax2.legend()
plt.tight_layout()
plt.show()


## 4. Experiment B (Module 1): Membership Inference Attack (MIA)
### Problem Formulation:
- **Target Fraud Task:** $f(x) \rightarrow y \in \{0, 1\}$
- **Attacker's Privacy Task:** $\mathcal{A}(x, y, f) \rightarrow m \in \{0, 1\}$  
  Determine whether transaction $(x, y)$ was included in the target training dataset ($m=1$) or held out ($m=0$).

### Protocol:
1. Strict 4-way disjoint split (Target Train, Target Holdout, Shadow Train, Shadow Holdout) prevents leakage.
2. Shadow model simulates target model behavior and outputs confidence features.
3. Attack classifier is trained on shadow model outputs and evaluated on the target model.
4. Baseline chance for a balanced 50/50 member/non-member test set is **50.0%**.


In [ ]:
from src.membership_inference import run_membership_inference_experiment

mia_results = run_membership_inference_experiment(X_train, y_train, save_outputs=True)
print("=== Membership Inference Attack Results ===")
print(f"Attack ROC-AUC       : {mia_results['attack_auc']}")
print(f"Attack Accuracy      : {mia_results['attack_accuracy']} (Random Chance Baseline = 0.5000)")
print(f"Attacker Advantage   : {mia_results['attacker_advantage']}")
print(f"Attack Precision     : {mia_results['attack_precision']}")
print(f"Attack Recall        : {mia_results['attack_recall']}")
print(f"Privacy Assessment   : {mia_results['leakage_assessment']}")


## 5. Experiment C (Module 2): Differential Privacy for Machine Learning
### Mathematical Foundation:
A randomized mechanism $\mathcal{M}$ satisfies $\epsilon$-Differential Privacy if:
$$P[\mathcal{M}(D) \in S] \le e^\epsilon P[\mathcal{M}(D') \in S]$$
- **IBM `diffprivlib` implementation:** Evaluates bounded sensitivity ($L_2$ clipping bound $C=5.0$) and calibrated objective perturbation.
- **Privacy-Utility Trade-off:** As privacy budget $\epsilon$ increases (from 0.1 to 10.0), noise decreases and classification utility rises.


In [ ]:
from src.differential_privacy import run_dp_privacy_utility_sweep

dp_sweep_df, dp_summary = run_dp_privacy_utility_sweep(
    X_train, y_train, X_test, y_test,
    epsilons=[0.1, 0.5, 1.0, 5.0, 10.0],
    data_norm=5.0,
    save_outputs=True
)

display(dp_sweep_df[["experiment_name", "epsilon", "precision", "recall", "f1_score", "pr_auc", "accuracy"]])


In [ ]:
# Privacy-Utility Pareto Frontier
plt.figure(figsize=(7, 4.2), dpi=120)
plt.plot(dp_sweep_df["epsilon"], dp_sweep_df["f1_score"], marker="o", color="#d62728", lw=2, label="DP Model F1-Score")
plt.plot(dp_sweep_df["epsilon"], dp_sweep_df["pr_auc"], marker="s", color="#1f77b4", lw=2, linestyle="--", label="DP Model PR-AUC")
plt.xscale("log")
plt.xlabel(r"Privacy Budget $\epsilon$ (Log Scale, Smaller = More Private)", fontweight="bold")
plt.ylabel("Evaluation Metric", fontweight="bold")
plt.title("Differential Privacy: Empirical Privacy-Utility Trade-Off", fontweight="bold")
plt.grid(True, which="both", linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Module 3: Local Differential Privacy (Client-Side Randomization)
### Conceptual Formulation:
- In Local DP, perturbation occurs **on the client device** before sending data to the server. No trusted central aggregator is assumed.
- We apply a bounded 1D Laplace mechanism to the transaction `Amount` feature:
  $$x' = x + \text{Laplace}\left(0, \frac{\Delta}{\epsilon_{loc}}\right)$$
- We measure feature distortion (Wasserstein distance) and downstream fraud classification degradation.


In [ ]:
from src.local_differential_privacy import run_ldp_demonstration

amount_idx = feature_names.index("Amount") if "Amount" in feature_names else 0
ldp_summary = run_ldp_demonstration(
    X_train, y_train, X_test, y_test,
    feature_index=amount_idx,
    feature_name="Amount",
    epsilons=[0.5, 1.0, 5.0],
    save_outputs=True
)

ldp_df = pd.DataFrame(ldp_summary["experiments"])
display(ldp_df[["experiment_name", "epsilon", "wasserstein_distance", "feature_mse", "f1_score", "pr_auc"]])
print("
Analytical Takeaway:", ldp_summary["analytical_takeaway"])


## 7. Module 4: Privacy-Preserving Data Mining (PPDM)
### Crucial Clarification: Why PCA is NOT Differential Privacy
1. **PCA is linear dimensionality reduction:** It projects data onto orthogonal variance-maximizing eigenvectors ($Z = X W$).
2. **Zero noise and zero privacy guarantee:** PCA has no sensitivity bounds and allows high-fidelity reconstruction ($X \approx Z W^T$).
3. **PPDM Demonstration:** We implement **Continuous Microaggregation** to enforce $k$-anonymity by clustering records into groups of size $k$ and releasing centroid representations.


In [ ]:
from src.privacy_preserving_mining import run_ppdm_demonstration

ppdm_summary = run_ppdm_demonstration(X_train, y_train, X_test, y_test, k_values=[5, 10], save_outputs=True)
ppdm_df = pd.DataFrame(ppdm_summary["experiments"])
display(ppdm_df[["experiment_name", "k_anonymity", "information_loss_pct", "precision", "recall", "f1_score"]])
print("
PCA vs DP Clarification:
", ppdm_summary["pca_vs_dp_clarification"])


## 8. Module 5: Compressive Privacy for Machine Learning
### Information-Theoretic Formulation:
$$\min_{P(Z|X)} [ I(Z; S) - \beta I(Z; Y) ] \quad \text{subject to } I(Z; X) \le R$$
- **Objective:** Compress representation $Z$ such that mutual information with sensitive attribute $S$ is minimized while mutual information with utility task $Y$ (fraud) is preserved.
- **Demonstration:** Evaluates downstream fraud utility vs. adversary ability to infer sensitive proxy attribute $S$ across latent dimensions $d \in [5, 10, 15]$.


In [ ]:
from src.compressive_privacy import run_compressive_privacy_demonstration

comp_summary = run_compressive_privacy_demonstration(
    X_train, y_train, X_test, y_test,
    feature_names=feature_names,
    latent_dimensions=[5, 10, 15],
    save_outputs=True
)

comp_df = pd.DataFrame(comp_summary["experiments"])
display(comp_df[["experiment_name", "latent_dimension", "f1_score", "adversary_sensitive_auc", "pr_auc"]])
print("
Academic Scope Statement:", comp_summary["implementation_scope_statement"])


## 9. Consolidated Evaluation Table & Academic Viva Summary


In [ ]:
from src.evaluation import save_metrics_summary

all_records = [baseline_metrics] + dp_summary["dp_sweep_results"] + ldp_summary["experiments"] + ppdm_summary["experiments"] + comp_summary["experiments"]
summary_table_df = save_metrics_summary(all_records)

display(summary_table_df[["Experiment", "Precision", "Recall", "F1-Score", "PR-AUC", "Privacy/Attack Metric", "Notes/Limitations"]])
print("Evaluation table persisted to results/experiments_comparison.csv")
